In [1]:
#source TFAvenv/bin/activate (launching and exiting the virtual environment containing the required modules, stored in the working directory)
#TFAvenv/bin/python your_script.py - for running a script in the virtual environment
#source deactivate

#import all the libraries needed
from import_dep import *
import importlib

# import the PMU import classes - one per (instrument x measurement type)
import PMU
importlib.reload(PMU)
from PMU import (Keithley_PUND, Keithley_Haoran_PUND, AixACCT_PUND,
                 AixACCT_DHM, AixACCT_Fatigue, merge_PMU)

# import the transform and plotting functions
import PMU_functions
importlib.reload(PMU_functions)
from PMU_functions import (extract_pund, extract_dhm, extract_fatigue,
                           plot_PUND, plot_PUND_polarisation, plot_DHM,
                           plot_fatigue, update_plot_string)

# import custom plot style
import plot_style
importlib.reload(plot_style)
from plot_style import set_plot_style

# Root directories for each instrument
root_KEI = '/Users/horatiocox/Desktop/RUG_postdoc/Experiments/Keithley/'
root_AIX = '/Users/horatiocox/Desktop/RUG_postdoc/Experiments/AixAcct/'

# Per-sample data and output folders
data_JT147   = root_KEI + 'JT147/Data'
output_JT147 = root_KEI + 'JT147/Output'

data_H03C5   = root_KEI + 'H03C5/Data/PUND'
output_H03C5 = root_KEI + 'H03C5/Output'

data_SM04    = root_AIX + 'SM04/Data'
output_SM04  = root_AIX + 'SM04/Output'

data_SM06    = root_KEI + 'SM06/Data'
output_SM06 = root_KEI + 'SM06/Output'

### Plotting and Output Variables
export_data = False        # True to write figures and summary CSVs to the Output folder
fig_format = 'svg'         # format of the exported figures
plot_transparency = True   # transparent figure backgrounds

fig_size = set_plot_style(export_data=export_data, use_tex=True, markersize=0.1)


# PMU / Ferroelectric Analysis --- 2 Terminal

Pulsed and ferroelectric measurements on capacitor structures: **PUND**, **DHM**
(dynamic hysteresis) and **fatigue**.

Three instruments are supported, each with its own import class. A class scans
one sample folder and picks up only the files belonging to its own measurement
type, so a folder holding a PundFile, a HysterFile and a FatigueFile is opened
as three separate objects.

| Measurement | Keithley 4200A | Keithley (Haoran script) | aixACCT TFAnalyzer |
|---|---|---|---|
| PUND    | `Keithley_PUND` | `Keithley_Haoran_PUND` | `AixACCT_PUND` |
| DHM     | -- | -- | `AixACCT_DHM` |
| Fatigue | -- | -- | `AixACCT_Fatigue` |

Every run is a `PMUdata` dataclass holding its own `metadata`, `raw_data`, the
processed `pund` / `dhm` loop and the derived `extracted_params`. Runs live in a
dict inside the container object, so `pund_JT147[0]` gets run 0 and
`for m in pund_JT147:` iterates over all of them.

**Constructing an object also processes it.** Each class runs `_load_data()`
then `_transform_data()`, so the moment an importer returns, every run already
carries its polarisation loop and its extracted parameters --- the plotting
functions below only read stored data. Anything that changes the transform
(`med_filt`, `p_pulse`, `centre_branches`, `loop`, `Pr_mode`) therefore belongs
on the **importer**, not the plot call; passing it to a plot call still works
and re-extracts, but then the object and the figure can disagree. Pass
`transform=False` to import without processing.

    pund_JT147[0].pund               # the processed loop, a DataFrame
    pund_JT147[0].extracted_params   # Pr, 2Pr, Vc, Ec, imprint, Qsw
    pund_JT147[0].instrument_params  # what the instrument itself reported

**Geometry.** Polarisation scales inversely with electrode area and the electric
field inversely with film thickness, so both must be right. aixACCT files carry
`Area [mm2]` and `Thickness [nm]` in their headers and the Haoran export carries
`area_cm2`; these are picked up automatically. Keithley Clarius records neither,
so pass `electrode_area` (in m^2) and `film_thickness_nm` yourself. A value
passed to the constructor always overrides whatever is in the file.


# PUND

The PUND (Positive--Up--Negative--Down) protocol separates the *switching*
polarisation of a ferroelectric from everything else the film does under bias.
Four pulses are applied:

$$ P\,(+V) \;\rightarrow\; U\,(+V) \;\rightarrow\; N\,(-V) \;\rightarrow\; D\,(-V) $$

$P$ and $N$ switch the polarisation and so carry switching **plus**
non-switching current; $U$ and $D$ follow immediately after, find the film
already poled, and so carry only the non-switching response --- leakage plus
linear dielectric charging. Each pair is the *same* waveform applied twice, so
subtracting them **pointwise in time-within-the-pulse** cancels everything but
the switching:

$$ I_\mathrm{sw}(t) = I_P(t) - I_U(t) \qquad
   I_\mathrm{sw}(t) = I_N(t) - I_D(t) $$

Integrating and normalising by the electrode area gives the polarisation, and
the remanent value follows from the switched charge:

$$ P(t) = \frac{1}{A}\int I_\mathrm{sw}(t)\,\mathrm{d}t
   \qquad\qquad P_r = \frac{Q_\mathrm{sw}}{2A} $$

A cumulative integral necessarily starts at zero, so an uncorrected branch runs
$0 \rightarrow 2P_r$. Each half is re-referenced about zero so that it runs
$-P_r \rightarrow +P_r$; that is what makes the two halves close into a loop.

**What PUND is and is not.** It reconstructs only the *switching* polarisation,
so the ramp-down of each branch is flat by construction. It is not a P--E
hysteresis loop --- that needs DHM or Sawyer--Tower --- but it gives the correct
$2P_r$ and coercive voltage, which a DHM loop does not, because a DHM loop
cannot be separated from leakage.

**What to look for**: two well-separated current peaks, one positive and one
negative, near $\pm V_c$, and a polarisation curve that saturates into flat
plateaux. A single broad hump, or plateaux that keep drifting, means leakage is
dominating rather than switching.


## Keithley 4200A --- PUND

The Clarius `nvm` pundTest module writes four trapezoidal pulses with **no
preset**, so the state of the film before the $P$ pulse depends on whatever ran
before it. When the two halves come out badly asymmetric, that is usually why:
the $N$--$D$ half is always well defined because $U$ leaves the film firmly
poled, whereas $P$--$U$ only measures true switching if the film happened to be
poled the other way to begin with.

Clarius records no device geometry, so set the electrode area and film thickness
here.


In [ ]:
## PUND import - Keithley 4200A, sample JT147

# Electrode geometry - CHECK THESE against the device that was actually probed
circ_diameter  = 40.0e-6                          # m, electrode diameter
electrode_area = np.pi * (circ_diameter / 2)**2   # m^2
film_thickness = 60.0                             # nm (None drops the E-field axis)

pund_JT147 = Keithley_PUND(data_JT147,
                           electrode_area=electrode_area,
                           film_thickness_nm=film_thickness,
                           med_filt=11,        # median filter on I before integrating
                           sample='JT147')

# Already processed - the parameters are available straight away
pd.DataFrame([m.extracted_params for m in pund_JT147],
             index=[m.plot_string for m in pund_JT147])


In [ ]:
## Waveforms vs time - the sanity check on the raw measurement
# Shows the pulse train as applied and the current it drew, before any subtraction.

fig_pund = plot_PUND([pund_JT147],
                     med_filt=11,
                     export_data=export_data,
                     output_PMU=output_JT147,
                     fig_name='JT147',
                     fig_format=fig_format,
                     plot_transparency=plot_transparency)


In [ ]:
## Switching polarisation loop
# Reads the loop the importer already built; nothing is recomputed here.

fig_pol, summary_JT147 = plot_PUND_polarisation(
    [pund_JT147],
    export_data=export_data,
    output_PMU=output_JT147,
    fig_name='JT147',
    fig_format=fig_format,
    plot_transparency=plot_transparency)

summary_JT147


## Keithley --- Haoran acquisition script

These workbooks arrive **already processed**: the `PUND_Diff` sheet holds the
difference current, charge, polarisation, branch and segment, so nothing is
recomputed --- only the derived scalars are filled in.

Two current ranges are recorded simultaneously. `channel='I2'` (the finer
range, and the one the script's own plots use) is the default; `channel='I1'`
selects the coarse range, which is worth checking if the fine range clipped.

The electrode area comes from the workbook's `Parameters` sheet. Thickness does
not, so pass it if you want field axes.

Note that this waveform carries a **+0.5 V offset**: the device rests at +0.5 V
between pulses rather than at zero. The voltages plotted are as measured, so
$V_c^+$ and $V_c^-$ are genuinely asymmetric and the reported imprint includes
that offset.


In [ ]:
## PUND import - Keithley, Haoran export, sample H03C5

pund_H03C5 = Keithley_Haoran_PUND(data_H03C5,
                                  film_thickness_nm=20.0,   # not stored in the file
                                  channel='I2',             # 'I1' for the coarse range
                                  sample='H03C5')

fig_H03C5, summary_H03C5 = plot_PUND_polarisation(
    [pund_H03C5],
    export_data=export_data,
    output_PMU=output_H03C5,
    fig_name='H03C5',
    fig_format=fig_format,
    plot_transparency=plot_transparency)

summary_H03C5


## aixACCT TFAnalyzer --- PUND

The aixACCT export writes **five** pulses, $P\,U\,N\,D\,P$: the fifth is a
second positive switching pulse. It is the one used by default, because it
follows $D$ and so the film is definitively down-poled before it, whereas the
first $P$ has no such guarantee. Pass `p_pulse='first'` to use the first
instead, or `p_pulse='both'` to plot them together and compare.

Each `Table N` in the file becomes one run. Area and thickness are read from the
file header. The instrument's own extracted values are kept separately in
`.instrument_params` so they can be compared against ours without either
overwriting the other:

* our `2Pr` is the **remanent** step, measured after the pulse returns to zero
* the instrument's `dPsw` = `Psw` - `Pnsw` is the swing measured **at $V_\max$**

Ours therefore tracks `dPsw` and sits slightly below it. A large gap, or a
`Pr` far beyond what the material can support, points at a current range that
clipped --- check `metadata['Current Range']`.


In [ ]:
## PUND import - aixACCT, sample SM04
# Note the file type is detected from its contents, not its name, so an export
# called NewPund.dat is picked up just as PundFile1.dat is.

pund_SM04 = AixACCT_PUND(data_SM04, sample='SM04',
                         p_pulse='last')   # 'first' or 'both' to compare

pund_SM04.summary_table()[['run', 'file', 'label', 'Current Range']]


In [ ]:
## Switching polarisation - the two well-conditioned tables
# Tables 1-2 were taken on the 100 mA range and are clipped; 3-4 are on 10 uA.

fig_SM04, summary_SM04 = plot_PUND_polarisation(
    [pund_SM04],
    run_nums=[2, 3],
    export_data=export_data,
    output_PMU=output_SM04,
    fig_name='SM04',
    fig_format=fig_format,
    plot_transparency=plot_transparency)

# Compare against what the instrument itself reported
pd.DataFrame([{'run': m.run_number,
               'table': m.metadata['table_no'],
               'our 2Pr': m.extracted_params['2Pr'],
               'instrument dPsw': m.instrument_params['dPsw'],
               'our Vc': m.extracted_params.get('Vc'),
               'our Ec (kV/cm)': m.extracted_params.get('Ec')}
              for m in pund_SM04.get([2, 3])[0]])


# Dynamic Hysteresis (DHM)

A DHM measurement sweeps a triangular voltage and records the full $P$--$V$
loop. Unlike PUND it cannot be separated into switching and non-switching
contributions, so a $P_r$ derived from it would silently include leakage and
dielectric charge. **The remanent and coercive values reported here are the
instrument's own**, copied from the file header. Use DHM for *shape* --- whether
the response is genuinely ferroelectric --- and PUND for a quantitative $P_r$.

Reading the plot, in the usual convention: polarisation on the left axis,
current on a twinned right axis with its zero aligned to $P = 0$.

* a **ferroelectric** gives a square loop and two sharp current peaks at
  $\pm V_c$
* a **leaky** film gives a rounded, slanted loop that opens up like an ellipse,
  with a broad current that tracks the voltage rather than peaking
* a **linear capacitor** gives a closed line with no hysteresis at all

The aixACCT export records three loops per table. Loop 1 is the main hysteresis
loop --- its first sample is exactly the reported $P_r^-$ --- while loops 2 and
3 are the relaxed-remanence loops ($P_{r,\mathrm{rel}}^-$ and
$P_{r,\mathrm{rel}}^+$), measured after a delay. Loop 3 is driven with the
opposite polarity; the right drive column is chosen automatically.


In [ ]:
## DHM import - aixACCT, sample SM04

dhm_SM04 = AixACCT_DHM(data_SM04, sample='SM04',
                       loop=1)   # 1 = main loop; 2/3 = relaxed remanence

fig_dhm, summary_dhm = plot_DHM(
    [dhm_SM04],
    run_nums=[0],
    export_data=export_data,
    output_PMU=output_SM04,
    fig_name='SM04',
    fig_format=fig_format,
    plot_transparency=plot_transparency)

summary_dhm


# Fatigue

A fatigue measurement cycles the film hard, stopping at a series of cycle counts
to measure a full loop each time. Each aixACCT `Result Table` becomes one
`FatigueRun`, holding

* `summary` --- the instrument's table of $P_r$, $V_c$ and the rest against
  cycle number
* `sub_runs` --- the loops themselves, one per cycle point, each an ordinary
  `PMUdata` object that the DHM and PUND plotting functions accept directly

**`Pr_mode`** selects where the plotted $P_r$ comes from. `'raw'` (the default)
uses the instrument's reported values. `'extracted'` re-derives $P_r$ from each
stored loop, which is only meaningful when the fatigue run interrogates the film
with PUND --- for a DHM-based run it warns and falls back to `'raw'`.

**`sub_plots=True`** writes one figure per cycle point into the output folder
without rendering any of them in the notebook. That is how to check whether a
rising $P_r$ is real switching or an artefact: a film that appears to gain
polarisation past $10^7$ cycles is usually breaking down, and the individual
loops will show it long before the summary curve does.

**`concatenate_stages=True`** treats successive runs as one continuous
experiment, offsetting each stage's cycle axis by the previous stage's final
count. It is off by default because aixACCT restarts the cycle count in every
Result Table, and whether the stages really are sequential on the same device is
a judgement about the experiment rather than about the file.


In [ ]:
## Fatigue import - aixACCT, sample SM04

# Pr_mode='extracted' recomputes Pr from each loop; only valid for PUND-based
# fatigue, and warns and falls back to 'raw' for a DHM-based run like this one.
fat_SM04 = AixACCT_Fatigue(data_SM04, sample='SM04', Pr_mode='raw')

fig_fat, summary_fat = plot_fatigue(
    [fat_SM04],
    export_data=export_data,
    output_PMU=output_SM04,
    fig_name='SM04',
    fig_format=fig_format,
    plot_transparency=plot_transparency)

summary_fat


In [ ]:
## Write every measured loop to disk, without rendering them here
# One figure per cycle point, into <output>/SM04_fatigue_loops/
# Needs export_data-independent output, so output_PMU must be set.

fig_fat, _ = plot_fatigue(
    [fat_SM04],
    sub_plots=True,
    output_PMU=output_SM04,
    fig_name='SM04',
    fig_format='png',           # png keeps 57 files manageable
    print_summary=False,
    plot_transparency=plot_transparency)


In [ ]:
## Inspect individual fatigue loops in the notebook
# sub_runs are ordinary PMUdata objects, so plot_DHM takes them directly.

run = fat_SM04[1]                      # the second Result Table
picks = [run.sub_runs[i] for i in (0, len(run.sub_runs) // 2, -1)]

fig_cmp, _ = plot_DHM(picks,
                      print_summary=False,
                      plot_transparency=plot_transparency)

# The cycle count each loop was measured at
[s.metadata['cycles'] for s in picks]


# Collected parameters

Everything derived is on the run objects, so a table across samples is just a
comprehension. `extracted_params` holds our values; `instrument_params` holds
whatever the instrument reported, untouched.


In [ ]:
## Parameters across every PUND run loaded above

rows = []
for obj in (pund_JT147, pund_H03C5, pund_SM04):
    for m in obj:
        rows.append({'sample': m.sample, 'instrument': m.instrument,
                     'run': m.run_number, 'label': m.plot_string,
                     **{k: v for k, v in m.extracted_params.items()
                        if isinstance(v, (int, float))}})

all_pund = pd.DataFrame(rows)
if export_data:
    all_pund.to_csv(Path(output_SM04) / 'PUND_all_samples.csv', index=False)
all_pund


In [ ]:
## Relabel runs for publication figures
# update_plot_string prints each run's index and current label; assign a new one
# with  pund_SM04[2].plot_string = 'as-grown'

update_plot_string([pund_JT147, pund_H03C5, pund_SM04])
